# Task 4: Transfer Learning Techniques

This notebook implements the required transfer learning section.

Implemented techniques:

1. **Sequential Fine-Tuning**  
   General dataset → Domain dataset

2. **Domain-Adaptive Pretraining (DAPT)**  
   Further pretrain BERT on domain text using Masked Language Modelling

3. **Few-Shot Fine-Tuning**  
   Fine-tune BERT using only a small portion of labelled domain data

These techniques demonstrate how transfer learning helps BERT adapt from general language understanding to domain-specific sentiment classification.


## Cell 1: Install Required Libraries

**Why we use this:**  
We need Hugging Face Transformers for BERT, PyTorch for model training, and scikit-learn for evaluation.


In [1]:
# !pip install transformers torch scikit-learn pandas numpy

## Cell 2: Import Libraries

**Why we use this:**  
These libraries allow us to load data, tokenize text, train Transformer models, and evaluate results.


In [ ]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    BertForMaskedLM,
    DataCollatorForLanguageModeling
)

from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader


/Users/finleyedmonds/Library/Mobile Documents/com~apple~CloudDocs/University /Swinburne/2026 Semester 1/COS60011 Technology Design Project/Assessments/Project Files/.venv2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup environment to use MPS

In [3]:
# Set MPS as default device
import torch

# Check MPS availability
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU")

# Set default tensor type for MPS
if device.type == "mps":
    torch.set_default_tensor_type('torch.FloatTensor')


Using MPS device


/Users/finleyedmonds/Library/Mobile Documents/com~apple~CloudDocs/University /Swinburne/2026 Semester 1/COS60011 Technology Design Project/Assessments/Project Files/.venv2/lib/python3.14/site-packages/torch/__init__.py:1331: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:436.)
  _C._set_default_tensor_type(t)


## Cell 3: Set Random Seed

**Why we use this:**  
Setting a random seed makes the experiment more repeatable.


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Cell 4: Load Datasets

**Dataset roles:**

- `goemotions_5class.csv` = general/source dataset
- `fpb_5class.csv` = domain/target dataset

**Why we use this:**  
Transfer learning needs a general dataset for broad sentiment knowledge and a domain dataset for adaptation.


In [5]:
general_df = pd.read_csv("goemotions_5class.csv")
domain_df = pd.read_csv("fpb_5class.csv")

print("General dataset shape:", general_df.shape)
print("Domain dataset shape:", domain_df.shape)

display(general_df.head())
display(domain_df.head())


General dataset shape: (43404, 3)
Domain dataset shape: (4846, 3)


,text,label,source
0,my favourite food is anything i didn't have to...,Neutral,GoEmotions
1,"now if he does off himself, everyone will thin...",Neutral,GoEmotions
2,why the fuck is bayless isoing,Fear,GoEmotions
3,to make her feel threatened,Fear,GoEmotions
4,dirty southern wankers,Fear,GoEmotions


,text,label,source
0,"according to gran , the company has no plans t...",Neutral,FinancialPhraseBank
1,technopolis plans to develop in stages an area...,Neutral,FinancialPhraseBank
2,the international electronic industry company ...,Sadness,FinancialPhraseBank
3,with the new production plant the company woul...,Optimism,FinancialPhraseBank
4,according to the company 's updated strategy f...,Optimism,FinancialPhraseBank


## Cell 5: Check Column Names

**Why we use this:**  
We must confirm the correct text and label columns before training.


In [6]:
print("General columns:", general_df.columns.tolist())
print("Domain columns:", domain_df.columns.tolist())


General columns: ['text', 'label', 'source']
Domain columns: ['text', 'label', 'source']


## Cell 6: Set Text and Label Columns

Change these only if your dataset columns are different.


In [7]:
text_column = "text"
label_column = "label"


## Cell 7: Clean Data

**Why we use this:**  
This removes missing rows and ensures the text column is in string format.


In [8]:
general_df = general_df[[text_column, label_column]].dropna()
domain_df = domain_df[[text_column, label_column]].dropna()

general_df[text_column] = general_df[text_column].astype(str)
domain_df[text_column] = domain_df[text_column].astype(str)

print("General dataset after cleaning:", general_df.shape)
print("Domain dataset after cleaning:", domain_df.shape)


General dataset after cleaning: (43404, 2)
Domain dataset after cleaning: (4846, 2)


## Cell 8: Convert Labels to Numeric Values

**Why we use this:**  
BERT requires numeric labels for classification.

This function works if your labels are already numbers or if they are text labels.


In [9]:
print("General labels before conversion:", general_df[label_column].unique())
print("Domain labels before conversion:", domain_df[label_column].unique())

label_map = {
    "neutral": 0,
    "optimism": 1,
    "sadness": 2,
    "joy": 3,
    "fear": 4
}

def convert_labels(df, label_column):
    if pd.api.types.is_numeric_dtype(df[label_column]):
        df[label_column] = df[label_column].astype(int)
        return df

    df[label_column] = df[label_column].astype(str).str.lower().str.strip()
    df[label_column] = df[label_column].map(label_map)
    df = df.dropna()
    df[label_column] = df[label_column].astype(int)
    return df

general_df = convert_labels(general_df, label_column)
domain_df = convert_labels(domain_df, label_column)

print("General dataset final shape:", general_df.shape)
print("Domain dataset final shape:", domain_df.shape)

print("\nGeneral label counts:")
print(general_df[label_column].value_counts())

print("\nDomain label counts:")
print(domain_df[label_column].value_counts())


General labels before conversion: <StringArray>
['Neutral', 'Fear', 'Joy', 'Optimism', 'Sadness']
Length: 5, dtype: str
Domain labels before conversion: <StringArray>
['Neutral', 'Sadness', 'Optimism', 'Joy', 'Fear']
Length: 5, dtype: str
General dataset final shape: (43404, 2)
Domain dataset final shape: (4846, 2)

General label counts:
label
0    17312
1     9302
3     7624
4     6520
2     2646
Name: count, dtype: int64

Domain label counts:
label
0    2879
1    1303
2     558
3      60
4      46
Name: count, dtype: int64


## Cell 9: Load BERT Tokenizer and Set Hyperparameters

**Why we use this:**  
The tokenizer converts raw text into BERT-compatible input IDs and attention masks.


In [10]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

if torch.cuda.is_available():
    torch.device("cuda")
elif torch.mps.is_available():
    torch.device("mps")
else:
    torch.device("cpu")

MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3
NUM_LABELS = 5

print("Using device:", device)


Using device: mps


## Cell 10: Classification Dataset Class

**Why we use this:**  
This dataset class prepares labelled text data for sentiment classification.


In [11]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = int(self.labels[index])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long)
        }


## Cell 11: Domain Text Dataset for DAPT

**Why we use this:**  
DAPT uses unlabelled domain text for masked language modelling, so it only needs text and not sentiment labels.


In [12]:
class DomainTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = list(texts)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_special_tokens_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "special_tokens_mask": encoding["special_tokens_mask"].flatten()
        }


## Cell 12: Training Function for Sentiment Classification

**Why we use this:**  
This function trains BERT for labelled sentiment classification.


In [13]:
def train_classification_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(dataloader)


## Cell 13: Evaluation Function

**Why we use this:**  
This gives the quantitative metrics required for comparing transfer learning techniques.


In [14]:
def evaluate_model(model, dataloader, device):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("\nClassification Report:\n")
    print(classification_report(true_labels, predictions, zero_division=0))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(true_labels, predictions))

    return accuracy, precision, recall, f1


## Cell 14: Prepare Domain Train and Validation Split

**Why we use this:**  
All techniques are evaluated on the same domain validation set for a fair comparison.


In [15]:
domain_train_texts, domain_val_texts, domain_train_labels, domain_val_labels = train_test_split(
    domain_df[text_column].values,
    domain_df[label_column].values,
    test_size=0.2,
    random_state=SEED,
    stratify=domain_df[label_column].values
)

domain_train_dataset = SentimentDataset(domain_train_texts, domain_train_labels, tokenizer, MAX_LEN)
domain_val_dataset = SentimentDataset(domain_val_texts, domain_val_labels, tokenizer, MAX_LEN)

domain_train_loader = DataLoader(domain_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
domain_val_loader = DataLoader(domain_val_dataset, batch_size=BATCH_SIZE)

print("Domain training samples:", len(domain_train_dataset))
print("Domain validation samples:", len(domain_val_dataset))


Domain training samples: 3876
Domain validation samples: 970


# Technique 1: Sequential Fine-Tuning

General dataset → Domain dataset

**Why we use this:**  
Sequential fine-tuning transfers general sentiment knowledge from a general dataset into the domain dataset.


In [16]:
general_train_texts, general_val_texts, general_train_labels, general_val_labels = train_test_split(
    general_df[text_column].values,
    general_df[label_column].values,
    test_size=0.2,
    random_state=SEED,
    stratify=general_df[label_column].values
)

general_train_dataset = SentimentDataset(general_train_texts, general_train_labels, tokenizer, MAX_LEN)
general_train_loader = DataLoader(general_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("General training samples:", len(general_train_dataset))


General training samples: 34723


In [17]:
model_sequential = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_sequential.to(device)

optimizer = AdamW(model_sequential.parameters(), lr=LEARNING_RATE)

print("Stage 1: Fine-tuning BERT on general dataset")

for epoch in range(EPOCHS):
    print(f"General Training Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_classification_model(model_sequential, general_train_loader, optimizer, device)
    print("General Training Loss:", train_loss)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9249.00it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Stage 1: Fine-tuning BERT on general dataset
General Training Epoch 1/3
General Training Loss: 0.8679463662677293
General Training Epoch 2/3
General Training Loss: 0.6608654926005513
General Training Epoch 3/3
General Training Loss: 0.4770443993970196


In [18]:
optimizer = AdamW(model_sequential.parameters(), lr=LEARNING_RATE)

print("Stage 2: Fine-tuning BERT on domain dataset")

for epoch in range(EPOCHS):
    print(f"Domain Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_classification_model(model_sequential, domain_train_loader, optimizer, device)
    print("Domain Fine-Tuning Loss:", train_loss)
    sequential_results = evaluate_model(model_sequential, domain_val_loader, device)


Stage 2: Fine-tuning BERT on domain dataset
Domain Fine-Tuning Epoch 1/3
Domain Fine-Tuning Loss: 0.495889935351203
Accuracy: 0.8350515463917526
Precision: 0.83237369458085
Recall: 0.8350515463917526
F1 Score: 0.8326442774475867

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.88      0.89       576
           1       0.75      0.77      0.76       261
           2       0.77      0.88      0.82       112
           3       0.43      0.25      0.32        12
           4       0.25      0.11      0.15         9

    accuracy                           0.84       970
   macro avg       0.62      0.58      0.59       970
weighted avg       0.83      0.84      0.83       970


Confusion Matrix:

[[505  51  16   3   1]
 [ 47 202  11   1   0]
 [  4   7  99   0   2]
 [  2   7   0   3   0]
 [  5   1   2   0   1]]
Domain Fine-Tuning Epoch 2/3
Domain Fine-Tuning Loss: 0.2617847756501824
Accuracy: 0.8268041237113402
Precision: 0.839584

# Technique 2: Domain-Adaptive Pretraining (DAPT)

DAPT further pretrains BERT on domain-specific text using Masked Language Modelling.

**Why we use this:**  
DAPT helps BERT learn domain vocabulary and writing patterns before sentiment classification.


In [19]:
dapt_dataset = DomainTextDataset(domain_df[text_column].values, tokenizer, MAX_LEN)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

dapt_loader = DataLoader(
    dapt_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

print("DAPT training samples:", len(dapt_dataset))


DAPT training samples: 4846


In [24]:
model_dapt_mlm = BertForMaskedLM.from_pretrained("bert-base-uncased")
model_dapt_mlm.to(device)

optimizer = AdamW(model_dapt_mlm.parameters(), lr=LEARNING_RATE)

print("Starting Domain-Adaptive Pretraining using Masked Language Modelling")

for epoch in range(1):
    model_dapt_mlm.train()
    total_loss = 0

    for batch in dapt_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_dapt_mlm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(dapt_loader)
    print(f"DAPT Epoch {epoch + 1}, Loss: {avg_loss}")


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11714.15it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Domain-Adaptive Pretraining using Masked Language Modelling
DAPT Epoch 1, Loss: 2.546962225791251


## Convert DAPT Model to Sentiment Classifier

**Why we use this:**  
After DAPT, the adapted BERT weights are transferred into a sentiment classification model.


In [27]:
model_dapt_classifier = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_dapt_classifier.bert.load_state_dict(model_dapt_mlm.bert.state_dict(), strict=False)

model_dapt_classifier.to(device)

optimizer = AdamW(model_dapt_classifier.parameters(), lr=LEARNING_RATE)

print("Fine-tuning DAPT-adapted BERT on labelled domain dataset")

for epoch in range(EPOCHS):
    print(f"DAPT Classifier Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_classification_model(model_dapt_classifier, domain_train_loader, optimizer, device)
    print("Training Loss:", train_loss)
    dapt_results = evaluate_model(model_dapt_classifier, domain_val_loader, device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12287.88it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Fine-tuning DAPT-adapted BERT on labelled domain dataset
DAPT Classifier Fine-Tuning Epoch 1/3
Training Loss: 0.7530882531478081
Accuracy: 0.8134020618556701
Precision: 0.8090951235929116
Recall: 0.8134020618556701
F1 Score: 0.8079108296006191

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.83      0.86       576
           1       0.68      0.84      0.75       261
           2       0.79      0.83      0.81       112
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00         9

    accuracy                           0.81       970
   macro avg       0.47      0.50      0.48       970
weighted avg       0.81      0.81      0.81       970


Confusion Matrix:

[[477  82  17   0   0]
 [ 36 219   6   0   0]
 [  8  11  93   0   0]
 [  1  11   0   0   0]
 [  7   1   1   0   0]]
DAPT Classifier Fine-Tuning Epoch 2/3
Training Loss: 0.36920170124175616
Accuracy: 0.8422680412371134
Prec

# Technique 3: Few-Shot Fine-Tuning

Only a small amount of labelled domain data is used for training.

**Why we use this:**  
Few-shot fine-tuning tests whether BERT can adapt when labelled domain data is limited.


In [28]:
few_shot_fraction = 0.1

few_shot_df, _ = train_test_split(
    pd.DataFrame({
        text_column: domain_train_texts,
        label_column: domain_train_labels
    }),
    train_size=few_shot_fraction,
    random_state=SEED,
    stratify=domain_train_labels
)

few_shot_dataset = SentimentDataset(
    few_shot_df[text_column].values,
    few_shot_df[label_column].values,
    tokenizer,
    MAX_LEN
)

few_shot_loader = DataLoader(few_shot_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Few-shot training samples:", len(few_shot_dataset))
print(few_shot_df[label_column].value_counts())


Few-shot training samples: 387
label
0    230
1    104
2     44
3      5
4      4
Name: count, dtype: int64


In [29]:
model_few_shot = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_few_shot.to(device)

optimizer = AdamW(model_few_shot.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    print(f"Few-Shot Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_classification_model(model_few_shot, few_shot_loader, optimizer, device)
    print("Few-Shot Training Loss:", train_loss)
    few_shot_results = evaluate_model(model_few_shot, domain_val_loader, device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12051.03it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Few-Shot Fine-Tuning Epoch 1/3
Few-Shot Training Loss: 1.1500781130790712
Accuracy: 0.5938144329896907
Precision: 0.3526155808268679
Recall: 0.5938144329896907
F1 Score: 0.4424800949573892

Classification Report:

              precision    recall  f1-score   support

           0       0.59      1.00      0.75       576
           1       0.00      0.00      0.00       261
           2       0.00      0.00      0.00       112
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00         9

    accuracy                           0.59       970
   macro avg       0.12      0.20      0.15       970
weighted avg       0.35      0.59      0.44       970


Confusion Matrix:

[[576   0   0   0   0]
 [261   0   0   0   0]
 [112   0   0   0   0]
 [ 12   0   0   0   0]
 [  9   0   0   0   0]]
Few-Shot Fine-Tuning Epoch 2/3
Few-Shot Training Loss: 0.8555962705612182
Accuracy: 0.654639175257732
Precision: 0.5513044540647943
Recall: 0.654639175257732
F1 

# Final Transfer Learning Comparison

**Why we use this:**  
This table compares the implemented transfer learning techniques quantitatively.


In [30]:
print("Final Evaluation: Sequential Fine-Tuning")
sequential_results = evaluate_model(model_sequential, domain_val_loader, device)

print("\nFinal Evaluation: DAPT + Domain Fine-Tuning")
dapt_results = evaluate_model(model_dapt_classifier, domain_val_loader, device)

print("\nFinal Evaluation: Few-Shot Fine-Tuning")
few_shot_results = evaluate_model(model_few_shot, domain_val_loader, device)

task4_results_df = pd.DataFrame({
    "Transfer Learning Technique": [
        "Sequential Fine-Tuning",
        "DAPT + Domain Fine-Tuning",
        "Few-Shot Fine-Tuning"
    ],
    "Accuracy": [
        sequential_results[0],
        dapt_results[0],
        few_shot_results[0]
    ],
    "Precision": [
        sequential_results[1],
        dapt_results[1],
        few_shot_results[1]
    ],
    "Recall": [
        sequential_results[2],
        dapt_results[2],
        few_shot_results[2]
    ],
    "F1 Score": [
        sequential_results[3],
        dapt_results[3],
        few_shot_results[3]
    ]
})

task4_results_df


Final Evaluation: Sequential Fine-Tuning
Accuracy: 0.8360824742268042
Precision: 0.8387792691716625
Recall: 0.8360824742268042
F1 Score: 0.8361576499623921

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.86      0.88       576
           1       0.75      0.83      0.79       261
           2       0.80      0.84      0.82       112
           3       0.43      0.25      0.32        12
           4       0.33      0.33      0.33         9

    accuracy                           0.84       970
   macro avg       0.64      0.62      0.63       970
weighted avg       0.84      0.84      0.84       970


Confusion Matrix:

[[494  58  18   3   3]
 [ 39 217   4   1   0]
 [  7   8  94   0   3]
 [  2   7   0   3   0]
 [  5   0   1   0   3]]

Final Evaluation: DAPT + Domain Fine-Tuning
Accuracy: 0.8443298969072165
Precision: 0.8376966193903759
Recall: 0.8443298969072165
F1 Score: 0.8404409457228516

Classification Report:

         

,Transfer Learning Technique,Accuracy,Precision,Recall,F1 Score
0,Sequential Fine-Tuning,0.836082,0.838779,0.836082,0.836158
1,DAPT + Domain Fine-Tuning,0.844330,0.837697,0.844330,0.840441
2,Few-Shot Fine-Tuning,0.745361,0.738880,0.745361,0.730940


## Save Best Transfer Learning Model

**Why we use this:**  
The best transfer learning model can be saved and reused later.


In [34]:
best_transfer_model = model_sequential

best_transfer_model.save_pretrained("best_task4_transfer_learning_model")
tokenizer.save_pretrained("best_task4_transfer_learning_model")

print("Best Task 4 transfer learning model saved successfully.")
evaluate_model(best_transfer_model, domain_val_loader, device)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.74it/s]


Best Task 4 transfer learning model saved successfully.
Accuracy: 0.8360824742268042
Precision: 0.8387792691716625
Recall: 0.8360824742268042
F1 Score: 0.8361576499623921

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.86      0.88       576
           1       0.75      0.83      0.79       261
           2       0.80      0.84      0.82       112
           3       0.43      0.25      0.32        12
           4       0.33      0.33      0.33         9

    accuracy                           0.84       970
   macro avg       0.64      0.62      0.63       970
weighted avg       0.84      0.84      0.84       970


Confusion Matrix:

[[494  58  18   3   3]
 [ 39 217   4   1   0]
 [  7   8  94   0   3]
 [  2   7   0   3   0]
 [  5   0   1   0   3]]


(0.8360824742268042,
 0.8387792691716625,
 0.8360824742268042,
 0.8361576499623921)

## Short Report Explanation

For Task 4, three transfer learning techniques were implemented. Sequential fine-tuning was used by first training BERT on a general dataset and then fine-tuning it on the domain dataset. Domain-Adaptive Pretraining was implemented by further pretraining BERT on domain-specific text using masked language modelling before sentiment classification. Few-shot fine-tuning was also tested by training BERT on only a small percentage of labelled domain data. These approaches were evaluated using accuracy, precision, recall, and F1-score to compare how different transfer learning methods improve domain adaptation.
